In [ ]:
# Check GPU VRAM usage
import torch

def show_gpu_memory():
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3  # Convert to GB
        reserved = torch.cuda.memory_reserved(0) / 1024**3
        allocated = torch.cuda.memory_allocated(0) / 1024**3
        free = total - reserved

        print(f"GPU: {gpu_name}")
        print(f"Total VRAM:     {total:.2f} GB")
        print(f"Reserved:       {reserved:.2f} GB")
        print(f"Allocated:      {allocated:.2f} GB")
        print(f"Free:           {free:.2f} GB")
    else:
        print("No GPU available")

show_gpu_memory()


In [ ]:
# Install required packages
%pip install fastapi uvicorn pyngrok transformers accelerate bitsandbytes sse-starlette -q


In [ ]:
# Load the LLM
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Using Qwen2.5-3B-Instruct - good reasoning + tool calling, fits easily on T4
model_id = "Qwen/Qwen2.5-3B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="cuda",  # Force GPU
    low_cpu_mem_usage=True,
)

print("Model loaded!")
show_gpu_memory()  # Check VRAM usage after loading


In [ ]:
# Create the API server (OpenAI-compatible with streaming & tool calling)
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
from typing import Optional, Any
import uvicorn
import re
import time
import uuid
import json
import asyncio

app = FastAPI(title="Qwen2.5-3B-Instruct API")

# ============== OpenAI-compatible Models ==============

class FunctionCall(BaseModel):
    name: str
    arguments: str  # JSON string

class ToolCall(BaseModel):
    id: str
    type: str = "function"
    function: FunctionCall

class Message(BaseModel):
    role: str
    content: Optional[str] = None
    tool_calls: Optional[list[ToolCall]] = None
    tool_call_id: Optional[str] = None  # For tool responses

class FunctionDef(BaseModel):
    name: str
    description: Optional[str] = None
    parameters: Optional[dict] = None

class Tool(BaseModel):
    type: str = "function"
    function: FunctionDef

class ChatCompletionRequest(BaseModel):
    model: str = "qwen2.5-3b-instruct"
    messages: list[Message]
    max_tokens: int = 8192
    temperature: float = 0.6
    top_p: float = 1.0
    stop: Optional[list[str]] = None  # Stop sequences
    stream: bool = False
    tools: Optional[list[Tool]] = None  # Available tools
    tool_choice: Optional[str] = "auto"  # "auto", "none", or {"type": "function", "function": {"name": "..."}}

class Choice(BaseModel):
    index: int
    message: Optional[Message] = None
    delta: Optional[dict] = None  # For streaming
    finish_reason: Optional[str] = None

class Usage(BaseModel):
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int

class ChatCompletionResponse(BaseModel):
    id: str
    object: str = "chat.completion"
    created: int
    model: str
    choices: list[Choice]
    usage: Optional[Usage] = None

# ============== Helper Functions ==============

def format_tools_for_prompt(tools: list[Tool]) -> str:
    """Format tools into a prompt section for the model."""
    if not tools:
        return ""

    tool_descriptions = []
    for tool in tools:
        func = tool.function
        tool_desc = f"- {func.name}: {func.description or 'No description'}"
        if func.parameters:
            tool_desc += f"\n  Parameters: {json.dumps(func.parameters)}"
        tool_descriptions.append(tool_desc)

    return f"""
You have access to the following tools:

{chr(10).join(tool_descriptions)}

To use a tool, respond with a JSON object in this format:
{{"tool_calls": [{{"name": "tool_name", "arguments": {{"arg1": "value1"}}}}]}}

Only use tools when necessary. If you can answer directly, do so.
"""

def parse_tool_calls(response: str) -> tuple[str, list[ToolCall]]:
    """Parse tool calls from model response."""
    tool_calls = []

    # Try to find JSON tool call format
    try:
        # Look for tool_calls JSON in response
        json_match = re.search(r'\{[^{}]*"tool_calls"[^{}]*\[.*?\]\s*\}', response, re.DOTALL)
        if json_match:
            data = json.loads(json_match.group())
            if "tool_calls" in data:
                for i, tc in enumerate(data["tool_calls"]):
                    tool_calls.append(ToolCall(
                        id=f"call_{uuid.uuid4().hex[:8]}",
                        type="function",
                        function=FunctionCall(
                            name=tc["name"],
                            arguments=json.dumps(tc.get("arguments", {}))
                        )
                    ))
                # Remove the JSON from response
                response = response[:json_match.start()] + response[json_match.end():]
    except (json.JSONDecodeError, KeyError):
        pass

    return response.strip(), tool_calls

def parse_thinking(response: str) -> str:
    """Remove thinking tags from response if present."""
    # Remove various thinking tag formats
    response = re.sub(r'\[THINK\].*?\[/THINK\]', '', response, flags=re.DOTALL | re.IGNORECASE)
    response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL | re.IGNORECASE)
    return response.strip()

# ============== Endpoints ==============

@app.get("/v1/models")
def list_models():
    return {
        "object": "list",
        "data": [{
            "id": "qwen2.5-3b-instruct",
            "object": "model",
            "created": int(time.time()),
            "owned_by": "local"
        }]
    }

@app.get("/")
def root():
    return {"status": "ok", "model": model_id}

@app.get("/health")
def health():
    return {"status": "healthy", "gpu_available": torch.cuda.is_available()}

# Streaming generator
async def generate_stream(request: ChatCompletionRequest, prompt_tokens: int):
    """Generate streaming response using Server-Sent Events."""
    messages = [{"role": m.role, "content": m.content or ""} for m in request.messages]

    # Add tool descriptions to system message if tools provided
    if request.tools:
        tool_prompt = format_tools_for_prompt(request.tools)
        if messages and messages[0]["role"] == "system":
            messages[0]["content"] += "\n" + tool_prompt
        else:
            messages.insert(0, {"role": "system", "content": tool_prompt})

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    request_id = f"chatcmpl-{uuid.uuid4().hex[:8]}"
    created = int(time.time())

    # Initial chunk
    yield f"data: {json.dumps({'id': request_id, 'object': 'chat.completion.chunk', 'created': created, 'model': request.model, 'choices': [{'index': 0, 'delta': {'role': 'assistant'}, 'finish_reason': None}]})}\n\n"

    # Generate tokens one by one
    generated_text = ""
    with torch.no_grad():
        past_key_values = None
        input_ids = inputs.input_ids

        for _ in range(request.max_tokens):
            outputs = model(input_ids, past_key_values=past_key_values, use_cache=True)
            past_key_values = outputs.past_key_values

            # Sample next token
            logits = outputs.logits[:, -1, :]
            if request.temperature > 0:
                logits = logits / request.temperature
                probs = torch.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = torch.argmax(logits, dim=-1, keepdim=True)

            # Check for EOS
            if next_token.item() == tokenizer.eos_token_id:
                break

            # Decode token
            token_text = tokenizer.decode(next_token[0], skip_special_tokens=True)
            generated_text += token_text

            # Check stop sequences
            if request.stop:
                should_stop = False
                for stop_seq in request.stop:
                    if stop_seq in generated_text:
                        generated_text = generated_text.split(stop_seq)[0]
                        should_stop = True
                        break
                if should_stop:
                    break

            # Yield chunk
            chunk = {
                "id": request_id,
                "object": "chat.completion.chunk",
                "created": created,
                "model": request.model,
                "choices": [{"index": 0, "delta": {"content": token_text}, "finish_reason": None}]
            }
            yield f"data: {json.dumps(chunk)}\n\n"

            input_ids = next_token
            await asyncio.sleep(0)  # Allow other tasks to run

    # Final chunk
    yield f"data: {json.dumps({'id': request_id, 'object': 'chat.completion.chunk', 'created': created, 'model': request.model, 'choices': [{'index': 0, 'delta': {}, 'finish_reason': 'stop'}]})}\n\n"
    yield "data: [DONE]\n\n"

@app.post("/v1/chat/completions")
async def chat_completions(request: ChatCompletionRequest):
    messages = [{"role": m.role, "content": m.content or ""} for m in request.messages]

    # Add tool descriptions if provided
    if request.tools and request.tool_choice != "none":
        tool_prompt = format_tools_for_prompt(request.tools)
        if messages and messages[0]["role"] == "system":
            messages[0]["content"] += "\n" + tool_prompt
        else:
            messages.insert(0, {"role": "system", "content": tool_prompt})

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    prompt_tokens = inputs.input_ids.shape[1]

    # Handle streaming
    if request.stream:
        return StreamingResponse(
            generate_stream(request, prompt_tokens),
            media_type="text/event-stream"
        )

    # Non-streaming generation
    stop_token_ids = []
    if request.stop:
        for stop_seq in request.stop:
            stop_ids = tokenizer.encode(stop_seq, add_special_tokens=False)
            if stop_ids:
                stop_token_ids.extend(stop_ids)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=request.max_tokens,
            temperature=request.temperature if request.temperature > 0 else 1.0,
            top_p=request.top_p,
            do_sample=request.temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=[tokenizer.eos_token_id] + stop_token_ids if stop_token_ids else tokenizer.eos_token_id,
        )

    completion_tokens = outputs.shape[1] - prompt_tokens
    full_response = tokenizer.decode(outputs[0][prompt_tokens:], skip_special_tokens=True)

    # Remove thinking sections
    response_text = parse_thinking(full_response)

    # Check for tool calls
    response_text, tool_calls = parse_tool_calls(response_text)

    # Build response
    response_message = Message(role="assistant", content=response_text if response_text else None)
    if tool_calls:
        response_message.tool_calls = tool_calls
        response_message.content = None  # OpenAI format: no content when tool_calls present

    finish_reason = "tool_calls" if tool_calls else "stop"

    return ChatCompletionResponse(
        id=f"chatcmpl-{uuid.uuid4().hex[:8]}",
        created=int(time.time()),
        model=request.model,
        choices=[Choice(index=0, message=response_message, finish_reason=finish_reason)],
        usage=Usage(
            prompt_tokens=prompt_tokens,
            completion_tokens=completion_tokens,
            total_tokens=prompt_tokens + completion_tokens
        )
    )

print("API server created (OpenAI-compatible with streaming & tools)!")


In [ ]:
# Start ngrok and the server
from pyngrok import ngrok
import threading

# Set your ngrok auth token (get one free at https://dashboard.ngrok.com)
ngrok.set_auth_token("36Rc4U0Tb3PUiX2vF3m99t31vvX_7XnhzuFKSp91EmFdXDXgA")

# Start ngrok tunnel
port = 8000
tunnel = ngrok.connect(port)
public_url = tunnel.public_url  # Extract just the URL string

print("=" * 60)
print("🚀 Your LLM API is now PUBLIC!")
print("=" * 60)
print(f"Public URL: {public_url}")
print()
print("Endpoints (OpenAI-compatible):")
print(f"  GET  {public_url}/v1/models           - List models")
print(f"  POST {public_url}/v1/chat/completions - Chat completions")
print(f"  GET  {public_url}/docs                - Interactive API docs")
print("=" * 60)
print()
print("Example: Basic chat")
print(f'curl -X POST "{public_url}/v1/chat/completions" \\')
print('  -H "Content-Type: application/json" \\')
print('  -d \'{"model": "qwen2.5-3b-instruct", "messages": [{"role": "user", "content": "Hello!"}]}\'')
print()
print("Example: Streaming")
print(f'curl -X POST "{public_url}/v1/chat/completions" \\')
print('  -H "Content-Type: application/json" \\')
print('  -d \'{"model": "qwen2.5-3b-instruct", "messages": [{"role": "user", "content": "Count to 5"}], "stream": true}\'')
print()
print("Example: With stop sequences")
print('  -d \'{"messages": [...], "stop": ["\\n\\n", "END"]}\'')
print()
print("Example: With tool calling")
print('  -d \'{"messages": [...], "tools": [{"type": "function", "function": {"name": "get_weather", "description": "Get weather for a city"}}]}\'')
print("=" * 60)
print()
print("Python client example:")
print(f'''
from openai import OpenAI

client = OpenAI(base_url="{public_url}/v1", api_key="not-needed")

# Basic chat
response = client.chat.completions.create(
    model="qwen2.5-3b-instruct",
    messages=[{{"role": "user", "content": "What is 2+2?"}}]
)

# Streaming
for chunk in client.chat.completions.create(
    model="qwen2.5-3b-instruct",
    messages=[{{"role": "user", "content": "Count to 5"}}],
    stream=True
):
    print(chunk.choices[0].delta.content or "", end="")

# Tool calling
response = client.chat.completions.create(
    model="qwen2.5-3b-instruct",
    messages=[{{"role": "user", "content": "What's the weather in Paris?"}}],
    tools=[{{
        "type": "function",
        "function": {{
            "name": "get_weather",
            "description": "Get current weather for a location",
            "parameters": {{"type": "object", "properties": {{"city": {{"type": "string"}}}}}}
        }}
    }}]
)
''')

# Run uvicorn in a background thread (needed for Jupyter/Colab)
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=port, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print()
print("✅ Server is running in background!")
print(f"📖 Visit {public_url}/docs for interactive API documentation")
